# VLA: от наблюдения к действию

Action chunks, маски, нормализация и семейства готовых политик.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

VLA получает наблюдение, инструкцию и, если архитектура поддерживает, состояние. Выход может быть токенами действий, вектором или chunk нескольких будущих действий. Loss зависит от политики: OpenVLA — action-token CE; ACT — reconstruction + KL; SmolVLA и π0 — flow matching.

Эти готовые политики в студии обучаются по демонстрациям, а не через универсальный GRPO/Unsloth. Поддержка RL собственных VLA — отдельный путь. Ниже только непрерывная masked imitation задача; это не ACT и не π0. Терминальный padding нельзя выдавать за реальные действия. Для робота дополнительно обязательны проверка единиц, calibration и ограничения действий.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
batch, horizon, action_dim = 24, 4, 3
state = torch.randn(batch, 6)
target = state[:, :3, None].transpose(1, 2).repeat(1, horizon, 1) * 0.2
target += torch.arange(horizon)[None, :, None] * 0.05
valid = torch.ones(batch, horizon, dtype=torch.bool); valid[:8, -2:] = False
mean = target[valid].mean(0); std = target[valid].std(0).clamp_min(1e-6)
normalized = (target - mean) / std
policy = nn.Sequential(nn.Linear(6, 32), nn.ReLU(), nn.Linear(32, horizon * action_dim))
optimizer = torch.optim.AdamW(policy.parameters(), lr=0.02)
for step in range(60):
    prediction = policy(state).reshape(batch, horizon, action_dim)
    loss = ((prediction - normalized).square() * valid[..., None]).sum() / (valid.sum() * action_dim)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
action = prediction.detach() * std + mean
print('valid steps:', valid.sum().item(), 'loss:', loss.item())
print('first chunk in original units:', action[0])

## Попробуйте сами

Замените horizon на 1 и 8. Уберите маску: как изменится смысл loss? Статистики нормализации в настоящей задаче считают только на train и сохраняют вместе с политикой.

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [OpenVLA](https://arxiv.org/abs/2406.09246)
- [ACT](https://arxiv.org/abs/2304.13705)
- [π0](https://arxiv.org/abs/2410.24164)